In [ ]:
!pip install yfinance

In [ ]:
import yfinance as yf
import pandas as pd

# List of tickers provided
tickers = ['VT', 'ASML', 'EWZ', 'XLC', 'XLP', 'XLV', 'XLK', 'VEA', 'GLD', 'BAESY', 'CEG', 'DOW', 'MS', 'SCHW', 'SETM', 'SHOP', 'SMFG']

# Download data for the last year
# We set auto_adjust=False to ensure 'Adj Close' column is present.
data = yf.download(tickers, period="1y", auto_adjust=False)['Adj Close']

# Display the first few rows of the data
display(data.head())

[*********************100%***********************]  17 of 17 completed


Ticker,ASML,BAESY,CEG,DOW,EWZ,GLD,MS,SCHW,SETM,SHOP,SMFG,VEA,VT,XLC,XLK,XLP,XLV
Date,,,,,,,,,,,,,,,,,
2025-02-24,729.573120,65.425980,266.228485,36.333916,23.768511,272.209991,126.573334,79.146202,14.852684,115.480003,14.958730,49.474331,118.969284,101.979271,115.038361,80.238266,145.068390
2025-02-25,723.853088,69.137123,262.837463,36.501396,23.977173,268.619995,126.212997,78.039810,14.630356,110.949997,15.185080,49.831928,118.831985,100.467667,113.547646,81.377167,146.316437
2025-02-26,740.447815,67.811707,268.634979,36.147823,23.597786,269.029999,127.625099,77.743454,14.805465,113.230003,15.155555,49.909245,119.008514,100.289841,114.784950,79.819710,145.264938
2025-02-27,691.645142,69.834183,248.348648,35.757042,23.436546,264.929993,125.862411,77.476738,14.549688,110.830002,15.263810,49.319691,117.213821,99.321609,110.605965,79.809967,144.675293
2025-02-28,702.926270,71.807571,249.149155,36.117172,22.867470,263.269989,129.631256,78.563370,14.284076,112.000000,15.066984,49.358356,118.243561,100.773933,112.066872,80.870987,146.355743


In [ ]:
# Calculate daily returns
daily_returns = data.pct_change()

# Display the first few rows of the daily returns
display(daily_returns.head())

Ticker,ASML,BAESY,CEG,DOW,EWZ,GLD,MS,SCHW,SETM,SHOP,SMFG,VEA,VT,XLC,XLK,XLP,XLV
Date,,,,,,,,,,,,,,,,,
2025-02-24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-02-25,-0.007840,0.056723,-0.012737,0.004609,0.008779,-0.013188,-0.002847,-0.013979,-0.014969,-0.039228,0.015132,0.007228,-0.001154,-0.014823,-0.012958,0.014194,0.008603
2025-02-26,0.022926,-0.019171,0.022057,-0.009687,-0.015823,0.001526,0.011188,-0.003798,0.011969,0.020550,-0.001944,0.001552,0.001486,-0.001770,0.010897,-0.019139,-0.007186
2025-02-27,-0.065910,0.029825,-0.075516,-0.010811,-0.006833,-0.015240,-0.013811,-0.003431,-0.017276,-0.021196,0.007143,-0.011813,-0.015080,-0.009654,-0.036407,-0.000122,-0.004059
2025-02-28,0.016311,0.028258,0.003223,0.010072,-0.024282,-0.006266,0.029944,0.014025,-0.018256,0.010557,-0.012895,0.000784,0.008785,0.014622,0.013208,0.013294,0.011615


In [ ]:
from scipy import stats
import pandas as pd

market_ticker = 'VT'
betas = {}

# Loop through each ticker to calculate beta
for ticker in daily_returns.columns:
    if ticker == market_ticker:
        continue  # Skip the market ticker itself

    # Select the specific stock and the market, dropping rows where either is NaN
    # This ensures we use all valid overlapping data points for each pair
    pair_data = daily_returns[[market_ticker, ticker]].dropna()

    if len(pair_data) < 2:
        betas[ticker] = None
        continue

    # Perform linear regression: Stock Returns (y) ~ Market Returns (x)
    slope, intercept, r_value, p_value, std_err = stats.linregress(pair_data[market_ticker], pair_data[ticker])

    betas[ticker] = slope

# Convert the results to a DataFrame for display
beta_df = pd.DataFrame.from_dict(betas, orient='index', columns=['Beta'])
beta_df.index.name = 'Ticker'

display(beta_df)

,Beta
Ticker,
ASML,1.691759
BAESY,0.506312
CEG,1.881280
DOW,1.791030
EWZ,0.824273
GLD,0.200233
MS,1.502464
SCHW,0.988724
SETM,1.498046


In [ ]:
# CAPM Parameters
risk_free_rate = 0.0364
market_risk_premium = 0.05

# Calculate Expected Return: Rf + Beta * (Rm - Rf)
beta_df['Expected Return'] = risk_free_rate + (beta_df['Beta'] * market_risk_premium)

# Display the DataFrame with Expected Returns
# Formatting the Expected Return column as a percentage for better readability
display(beta_df.style.format({'Expected Return': '{:.2%}'}))

,Beta,Expected Return
Ticker,,
ASML,1.691759,12.10%
BAESY,0.506312,6.17%
CEG,1.881280,13.05%
DOW,1.791030,12.60%
EWZ,0.824273,7.76%
GLD,0.200233,4.64%
MS,1.502464,11.15%
SCHW,0.988724,8.58%
SETM,1.498046,11.13%


In [ ]:
# Calculate the covariance matrix of daily returns
cov_matrix = daily_returns.cov()

# Display the covariance matrix (Required for optimization math)
# print("--- Covariance Matrix (Risk Calculation Engine) ---")
# display(cov_matrix.style.background_gradient(cmap='coolwarm', axis=None).format("{:.2e}"))

# Calculate the correlation matrix
corr_matrix = daily_returns.corr()

# Display the correlation matrix (Easier for humans to read)
print("\n--- Correlation Matrix (Diversification View) ---")
display(corr_matrix.style.background_gradient(cmap='coolwarm', axis=None).format("{:.2f}"))


--- Correlation Matrix (Diversification View) ---


Ticker,ASML,BAESY,CEG,DOW,EWZ,GLD,MS,SCHW,SETM,SHOP,SMFG,VEA,VT,XLC,XLK,XLP,XLV
Ticker,,,,,,,,,,,,,,,,,
ASML,1.00,0.08,0.53,0.34,0.41,0.06,0.64,0.43,0.47,0.47,0.45,0.61,0.70,0.55,0.75,0.04,0.29
BAESY,0.08,1.00,0.03,0.07,0.26,0.19,0.15,0.17,0.20,0.09,0.23,0.36,0.23,0.14,0.13,0.17,0.14
CEG,0.53,0.03,1.00,0.22,0.27,0.05,0.57,0.41,0.38,0.46,0.37,0.47,0.61,0.51,0.69,0.01,0.16
DOW,0.34,0.07,0.22,1.00,0.39,0.04,0.41,0.27,0.35,0.27,0.38,0.55,0.59,0.44,0.45,0.39,0.48
EWZ,0.41,0.26,0.27,0.39,1.00,0.24,0.39,0.31,0.47,0.31,0.45,0.63,0.58,0.45,0.48,0.25,0.27
GLD,0.06,0.19,0.05,0.04,0.24,1.00,0.04,0.04,0.52,0.02,0.15,0.29,0.13,-0.01,0.07,0.03,0.02
MS,0.64,0.15,0.57,0.41,0.39,0.04,1.00,0.73,0.46,0.64,0.55,0.65,0.81,0.72,0.78,0.20,0.40
SCHW,0.43,0.17,0.41,0.27,0.31,0.04,0.73,1.00,0.40,0.46,0.44,0.54,0.65,0.58,0.61,0.21,0.32
SETM,0.47,0.20,0.38,0.35,0.47,0.52,0.46,0.40,1.00,0.38,0.42,0.58,0.58,0.40,0.56,0.11,0.23


In [ ]:
from scipy.optimize import minimize
import numpy as np

# Prepare data for optimization
# We use the stocks for which we calculated Beta and Expected Return
tickers_list = beta_df.index.tolist()
exp_returns = beta_df['Expected Return']

# Align covariance matrix to the specific list of tickers and annualize it (approx 252 trading days)
cov_mat_annual = cov_matrix.loc[tickers_list, tickers_list] * 252

num_assets = len(tickers_list)

# Function to calculate portfolio statistics
def portfolio_performance(weights, mean_returns, cov_matrix):
    # Expected portfolio return
    returns = np.sum(mean_returns * weights)
    # Expected portfolio volatility
    std_dev = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return returns, std_dev

# Objective function: Minimize Negative Sharpe Ratio
def neg_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate):
    p_ret, p_std = portfolio_performance(weights, mean_returns, cov_matrix)
    return -(p_ret - risk_free_rate) / p_std

# Constraints: Sum of weights = 100%
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})

# Bounds: 0% <= Weight <= 100% for each stock
bounds = tuple((0, 1) for _ in range(num_assets))

# Initial Guess (Equal weights)
init_guess = num_assets * [1. / num_assets,]

# Perform Optimization (Unconstrained)
opt_results = minimize(neg_sharpe_ratio, init_guess,
                       args=(exp_returns, cov_mat_annual, risk_free_rate),
                       method='SLSQP', bounds=bounds, constraints=constraints)

# Extract Optimal Weights
unconstrained_weights = opt_results.x

# Calculate performance of optimal portfolio
unconstrained_ret, unconstrained_vol = portfolio_performance(unconstrained_weights, exp_returns, cov_mat_annual)
unconstrained_sharpe = -opt_results.fun

# For backward compatibility with other cells
opt_ret = unconstrained_ret
opt_vol = unconstrained_vol
opt_sharpe = unconstrained_sharpe
optimal_weights = unconstrained_weights

# Create a DataFrame for the results
unconstrained_portfolio = pd.DataFrame({'Ticker': tickers_list, 'Weight': unconstrained_weights})
unconstrained_portfolio['Weight'] = unconstrained_portfolio['Weight'].apply(lambda x: round(x, 4))

# Display Results
print("--- Unconstrained Max Sharpe Portfolio (Global Benchmark) ---")
print(f"Max Sharpe Ratio: {unconstrained_sharpe:.4f}")
print(f"Expected Annual Return: {unconstrained_ret:.2%}")
print(f"Expected Annual Volatility: {unconstrained_vol:.2%}")

# Show only tickers with significant weights (> 0.1%)
display(unconstrained_portfolio[unconstrained_portfolio['Weight'] > 0.001].sort_values(by='Weight', ascending=False).style.format({'Weight': '{:.2%}'}))

--- Unconstrained Max Sharpe Portfolio (Global Benchmark) ---
Max Sharpe Ratio: 0.2931
Expected Annual Return: 8.90%
Expected Annual Volatility: 17.95%


,Ticker,Weight
11,VEA,37.59%
13,XLK,26.71%
12,XLC,11.86%
14,XLP,7.61%
15,XLV,6.44%
6,MS,3.76%
3,DOW,2.15%
7,SCHW,1.68%
9,SHOP,1.18%
2,CEG,0.42%


In [ ]:
# Calculate VT (Market) Metrics
vt_volatility = daily_returns['VT'].std() * np.sqrt(252)
vt_expected_return = risk_free_rate + market_risk_premium # Beta is 1.0

# Calculate Sharpe Ratio
vt_sharpe = (vt_expected_return - risk_free_rate) / vt_volatility

print(f"--- VT (Global Market Benchmark) ---")
print(f"Expected Annual Return:     {vt_expected_return:.2%}")
print(f"Expected Annual Volatility: {vt_volatility:.2%}")
print(f"Sharpe Ratio:               {vt_sharpe:.4f}")

--- VT (Global Market Benchmark) ---
Expected Annual Return:     8.64%
Expected Annual Volatility: 16.96%
Sharpe Ratio:               0.2949


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np

# --- Get Latest Prices ---
# Fetch the most recent closing prices from the downloaded data
latest_prices = data.iloc[-1]

# --- Input User Holdings (Shares) ---
# UPDATE THESE VALUES with your actual share counts
# Currently initialized to match your previous portfolio values
current_shares = {
    'ASML': 41455.21 / latest_prices['ASML'],
    'EWZ': 42838.30 / latest_prices['EWZ'],
    'XLC': 29921.28 / latest_prices['XLC'],
    'XLP': 98358.75 / latest_prices['XLP'],
    'XLV': 64091.20 / latest_prices['XLV'],
    'XLK': 26376.40 / latest_prices['XLK'],
    'VEA': 20511.00 / latest_prices['VEA'],
    'GLD': 52774.39 / latest_prices['GLD']
}

# Treasuries/Cash (Fixed Value)
treasuries_value = 349826.64

# --- Calculate Portfolio Values ---
# Value = Shares * Price
current_holdings_data = {ticker: shares * latest_prices[ticker] for ticker, shares in current_shares.items()}
current_holdings_data['Treasuries'] = treasuries_value

# Calculate Total Portfolio Value
total_portfolio_value = sum(current_holdings_data.values())

print(f"Total Portfolio Value: ${total_portfolio_value:,.2f}")
print(f"Current Treasuries: ${current_holdings_data['Treasuries']:,.2f}")
print("-" * 30)

# --- Widget Setup ---
sliders = {}
view_sliders = {}
ui_rows = []

style = {'description_width': 'initial'}

# Define groups
current_portfolio_tickers = ['ASML', 'EWZ', 'XLC', 'XLP', 'XLV', 'XLK', 'VEA', 'GLD']
new_additions_tickers = ['BAESY', 'CEG', 'DOW', 'MS', 'SCHW', 'SETM', 'SHOP', 'SMFG']

# --- Section 0: Reserves ---
ui_rows.append(widgets.HTML("<h3>Reserves</h3>"))
treasury_widget = widgets.FloatText(
    value=50000.0,
    description='Reserved Cash ($):',
    style=style,
    step=1000.0
)
ui_rows.append(treasury_widget)

# --- Section 1: Return Views (New) ---
ui_rows.append(widgets.HTML("<h3>Return Expectations (Views)</h3>"))
ui_rows.append(widgets.Label("Adjust based on how much you think the stock is undervalued. Adds a flat boost (MRP * View%)."))

# Grid layout for views to save space
view_items = []
for ticker in tickers_list:
    v_slider = widgets.FloatSlider(
        value=0.0, min=-0.5, max=2.0, step=0.1,
        description=f'{ticker} Undervalued %',
        style=style, layout=widgets.Layout(width='95%')
    )
    view_sliders[ticker] = v_slider
    view_items.append(v_slider)

# Arrange in 2 columns
view_grid = widgets.GridBox(view_items, layout=widgets.Layout(grid_template_columns="repeat(2, 50%)"))

# Put inside an accordion to keep UI clean
view_accordion = widgets.Accordion(children=[view_grid])
view_accordion.set_title(0, 'Adjust Stock Views')
ui_rows.append(view_accordion)

# --- Section 2: Current Portfolio (Rebalance Constraints) ---
ui_rows.append(widgets.HTML("<h3>Current Portfolio (Rebalance Constraints)</h3>"))
ui_rows.append(widgets.Label("Defaults are locked to current weights (Min=Max). Widen range to allow trading."))

# Calculate Current Weights for Defaults
# Est Invested Capital based on default reserves
est_invested_capital = total_portfolio_value - treasury_widget.value
current_weights_pct = {}
for ticker in current_portfolio_tickers:
    val = current_holdings_data.get(ticker, 0.0)
    w = val / est_invested_capital if est_invested_capital > 0 else 0
    current_weights_pct[ticker] = w

constraint_sliders = {}
constraint_items = []

for ticker in current_portfolio_tickers:
    cw = current_weights_pct.get(ticker, 0.0)
    # Range Slider locked to current weight
    c_slider = widgets.FloatRangeSlider(
        value=(cw, cw),
        min=0.0, max=1.0, step=0.001,
        description=f'{ticker} ({cw:.1%})',
        style=style, layout=widgets.Layout(width='95%')
    )
    constraint_sliders[ticker] = c_slider
    constraint_items.append(c_slider)

# Grid Layout for Constraints
constraint_grid = widgets.GridBox(constraint_items, layout=widgets.Layout(grid_template_columns="repeat(2, 50%)"))
ui_rows.append(constraint_grid)

ui_rows.append(widgets.Label("Optimized Weights (Result):"))
# Result sliders (Disabled, for display)
for ticker in current_portfolio_tickers:
    w_slider = widgets.FloatSlider(
        value=0.0, min=0.0, max=1.0, step=1e-4,
        description=f'{ticker} Weight %',
        style=style, layout=widgets.Layout(width='50%'),
        readout_format='.2%',
        disabled=True
    )
    sliders[ticker] = w_slider
    ui_rows.append(w_slider)

# --- Section 3: New Additions (Global Constraints) ---
ui_rows.append(widgets.HTML("<h3>New Additions (Unified Constraints)</h3>"))

new_min_slider = widgets.FloatSlider(
    value=0.02, min=0.0, max=1.0, step=0.01,
    description='New Stock Min %',
    style=style, layout=widgets.Layout(width='45%')
)
new_max_slider = widgets.FloatSlider(
    value=0.15, min=0.0, max=1.0, step=0.01,
    description='New Stock Max %',
    style=style, layout=widgets.Layout(width='45%')
)

ui_rows.append(widgets.HBox([new_min_slider, new_max_slider]))

# Run Button
run_button = widgets.Button(
    description="Optimize Portfolio",
    button_style='success',
    layout=widgets.Layout(width='90%', margin='20px 0px 0px 0px')
)

# Output Area
out = widgets.Output()

# --- Optimization Function ---
def run_optimization_dashboard(b):
    with out:
        clear_output()

        # 1. Invested Capital Logic
        reserved_cash = treasury_widget.value
        invested_capital = total_portfolio_value - reserved_cash

        if invested_capital <= 0:
             print(f"Error: Reserved cash (${reserved_cash:,.2f}) exceeds Total Portfolio Value.")
             return

        print(f"Invested Capital: ${invested_capital:,.2f}")
        print("-" * 30)

        # 2. Recalculate Expected Returns based on Views
        adj_exp_returns = exp_returns.copy()

        print("Adjusted Return Expectations (Beta Neutral):")

        for ticker in tickers_list:
            beta_val = beta_df.loc[ticker, 'Beta']
            view_pct = view_sliders[ticker].value
            orig_er = exp_returns[ticker]

            if view_pct == 0:
                continue

            # Beta Neutral Logic
            # Formula: CAPM_ER + (MRP * View)
            flat_boost = market_risk_premium * view_pct
            new_er = orig_er + flat_boost

            adj_exp_returns[ticker] = new_er
            print(f"  -> {ticker}: View {view_pct:+.0%} | New Exp Return: {new_er:.2%} (was {orig_er:.2%})")

        print("-" * 30)

        # 3. Bounds & Sliders
        current_bounds = []
        global_min = new_min_slider.value
        global_max = new_max_slider.value

        for ticker in tickers_list:
            if ticker in current_portfolio_tickers:
                # Use per-ticker constraints from RangeSliders
                if ticker in constraint_sliders:
                    current_bounds.append(constraint_sliders[ticker].value)
                else:
                    current_bounds.append((0, 1))
            elif ticker in new_additions_tickers:
                current_bounds.append((global_min, global_max))
            else:
                current_bounds.append((0, 1))

        current_bounds = tuple(current_bounds)

        # Validate Bounds
        total_min_weight = sum([b[0] for b in current_bounds])
        if total_min_weight > 1.0001:
            print(f"Error: Min weights ({total_min_weight:.2%}) > 100%.")
            return

        # 4. Run Optimization with ADJUSTED Returns
        current_constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1.0})
        current_init_guess = [1.0 / num_assets] * num_assets

        try:
            opt_results = minimize(neg_sharpe_ratio, current_init_guess,
                                   args=(adj_exp_returns, cov_mat_annual, risk_free_rate),
                                   method='SLSQP', bounds=current_bounds, constraints=current_constraints)

            if not opt_results.success:
                print(f"Optimization failed: {opt_results.message}")
                return

            optimal_weights = opt_results.x

            # Update Result Sliders
            for i, ticker in enumerate(tickers_list):
                if ticker in sliders:
                    sliders[ticker].value = optimal_weights[i]

            # Metrics (using Adjusted Returns for the 'Expected' view)
            equity_ret, equity_vol = portfolio_performance(optimal_weights, adj_exp_returns, cov_mat_annual)
            equity_sharpe = (equity_ret - risk_free_rate) / equity_vol

            print(f"Max Sharpe Ratio (Adj): {equity_sharpe:.4f}")
            print(f"Expected Annual Return (Adj): {equity_ret:.2%}")
            print(f"Expected Annual Volatility: {equity_vol:.2%}")
            print("-" * 30)

            # Display Data
            dash_portfolio = pd.DataFrame({'Ticker': tickers_list, 'Weight': optimal_weights})
            dash_portfolio['Value ($)'] = dash_portfolio['Weight'] * invested_capital
            dash_portfolio['Price ($)'] = dash_portfolio['Ticker'].map(latest_prices)
            dash_portfolio['Shares'] = dash_portfolio['Value ($)'] / dash_portfolio['Price ($)']

            # Add Beta to DataFrame
            dash_portfolio['Beta'] = dash_portfolio['Ticker'].map(beta_df['Beta']) # Fixed Mapping

            final_df = dash_portfolio[dash_portfolio['Weight'] > 0.0001].sort_values(by='Weight', ascending=False)

            def highlight_new(row):
                if row['Ticker'] in new_additions_tickers:
                    return ['background-color: #ffffcc'] * len(row)
                return [''] * len(row)

            display(final_df.style.apply(highlight_new, axis=1).format({
                'Weight': '{:.2%}',
                'Value ($)': '${:,.2f}',
                'Price ($)': '${:,.2f}',
                'Shares': '{:.2f}',
                'Beta': '{:.2f}'
            }))

            # --- PLOTTING INTEGRATION ---
            print("Generating Graph (Equity Portfolio)...")

            asset_vols = np.sqrt(np.diag(cov_mat_annual))

            # Calculate VT Metrics (Market Benchmark)
            vt_vol = daily_returns['VT'].std() * np.sqrt(252)
            vt_ret = risk_free_rate + market_risk_premium # Beta=1 assumption

            # Benchmark Points (Using VT as Max Sharpe)
            rf = risk_free_rate
            ms_ret_g = vt_ret
            ms_vol_g = vt_vol

            lev2_ret = rf + 2 * (ms_ret_g - rf)
            lev2_vol = 2 * ms_vol_g
            lev3_ret = rf + 3 * (ms_ret_g - rf)
            lev3_vol = 3 * ms_vol_g

            plt.figure(figsize=(14, 8))

            # 1. Plot Original CAPM (Gray)
            plt.scatter(asset_vols, exp_returns, c='gray', label='Original CAPM', s=30, alpha=0.4)

            # 2. Plot Adjusted Views (Blue)
            plt.scatter(asset_vols, adj_exp_returns, c='steelblue', label='Adjusted Views', s=50, alpha=0.8)

            # 3. Draw Arrows and Labels
            for i, ticker in enumerate(tickers_list):
                vol = asset_vols[i]
                orig_ret = exp_returns[ticker]
                new_ret = adj_exp_returns[ticker]

                # Label the main point (New)
                plt.annotate(ticker, (vol, new_ret), xytext=(5, 5), textcoords='offset points', fontsize=8, alpha=0.8)

                # Draw arrow if different
                if abs(new_ret - orig_ret) > 0.0001:
                    plt.annotate('', xy=(vol, new_ret), xytext=(vol, orig_ret),
                                 arrowprops=dict(arrowstyle="->", color='red', lw=1.5))

            # Plot CAL (Anchored to VT)
            cal_x = [0, ms_vol_g, lev2_vol, lev3_vol]
            cal_y = [rf, ms_ret_g, lev2_ret, lev3_ret]
            plt.plot(cal_x, cal_y, linestyle='--', color='crimson', linewidth=2, label='Capital Allocation Line (VT)', alpha=0.7)

            # Plot Specific Points
            plt.scatter([0], [rf], color='green', s=100, label='Risk-Free Rate', zorder=5, edgecolors='black')
            plt.scatter([ms_vol_g], [ms_ret_g], color='gold', s=300, marker='*', label='VT (Market Benchmark)', zorder=10, edgecolors='black')

            # PLOT THE NORMALIZED CONSTRAINED PORTFOLIO
            plt.scatter([equity_vol], [equity_ret], color='cyan', s=300, marker='*', label='Constrained Portfolio (Adj Views)', zorder=10, edgecolors='black')

            plt.scatter([lev2_vol, lev3_vol], [lev2_ret, lev3_ret], color='purple', s=80, marker='D', label='2x/3x Leveraged (Ref)', alpha=0.5)

            plt.title('Portfolio Optimization: Equity Only', fontsize=16)
            plt.xlabel('Annualized Standard Deviation (Risk)', fontsize=12)
            plt.ylabel('Expected Annual Return (Adjusted)', fontsize=12)
            plt.legend(loc='upper left', fontsize=10)
            plt.grid(True, linestyle='--', alpha=0.5)
            plt.xlim(left=0)

            plt.show()

        except Exception as e:
            print(f"Error: {e}")

run_button.on_click(run_optimization_dashboard)

dashboard = widgets.VBox([
    widgets.Label("Portfolio Optimizer with Views"),
    widgets.VBox(ui_rows),
    widgets.Box([run_button], layout=widgets.Layout(justify_content='center')),
    out
])
display(dashboard)

Total Portfolio Value: $726,153.17
Current Treasuries: $349,826.64
------------------------------
